# Orchestrator-Subagent — LangChain

**Pattern:** Orchestrator plans → delegates to specialist chains

```
Orchestrator
  ├── highlights_agent
  ├── logistics_agent
  ├── itinerary_agent
  └── package_formatter
```

The orchestrator function calls specialists in sequence. Each specialist is a focused LCEL chain. The orchestrator is plain Python — no framework magic.

In [ ]:
import os
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

load_dotenv()
assert os.getenv("GOOGLE_API_KEY"), "Set GOOGLE_API_KEY in .env"
llm = ChatGoogleGenerativeAI(model="gemini-2.0-flash", temperature=0)
parser = StrOutputParser()
print("✓ Ready")

In [ ]:
HIGHLIGHTS = {"tokyo": "Shibuya, Senso-ji, Tsukiji, Mt Fuji", "paris": "Eiffel, Louvre, Notre Dame", "bangalore": "Lalbagh, Nandi Hills"}
LOGISTICS  = {"tokyo": "3h from SFO. Shinkansen. Best: Mar-May.", "paris": "11h from NYC. Metro. Best: Apr-Jun.", "bangalore": "9h from Dubai. Ola. Best: Oct-Feb."}

highlights_agent = (ChatPromptTemplate.from_messages([("system","List top 3 attractions with 1-line descriptions."),("human","City: {city}\nData: {data}")]) | llm | parser)
logistics_agent  = (ChatPromptTemplate.from_messages([("system","Provide practical travel tips: flights, transport, season."),("human","City: {city}\nData: {data}")]) | llm | parser)
itinerary_agent  = (ChatPromptTemplate.from_messages([("system","Create a 3-day itinerary."),("human","City: {city}\nHighlights: {highlights}\nLogistics: {logistics}")]) | llm | parser)
formatter        = (ChatPromptTemplate.from_messages([("system","Format into polished trip package with ## sections."),("human","City: {city}\nHighlights: {highlights}\nLogistics: {logistics}\nItinerary: {itinerary}")]) | llm | parser)
print("4 specialist chains ready")

In [ ]:
def orchestrate(city: str) -> str:
    print(f"[Orchestrator] {city}")
    h = highlights_agent.invoke({"city": city, "data": HIGHLIGHTS.get(city.lower(),"N/A")})
    print("  → highlights done")
    l = logistics_agent.invoke({"city": city, "data": LOGISTICS.get(city.lower(),"N/A")})
    print("  → logistics done")
    i = itinerary_agent.invoke({"city": city, "highlights": h, "logistics": l})
    print("  → itinerary done")
    pkg = formatter.invoke({"city": city, "highlights": h, "logistics": l, "itinerary": i})
    print("  → package formatted")
    return pkg

package = orchestrate("Tokyo")
print("\n" + "="*50)
print(package)

## Key Takeaways

| What You Saw | Why It Matters |
|---|---|
| Orchestrator as a Python function | Orchestration = explicit code — full control |
| Each specialist only sees its inputs | No shared state — specialists are isolated |
| Orchestrator sequences calls | Order encoded in function body |

### Next: [LangGraph Orchestrator →](../LangGraph/orchestrator.ipynb)